# Embedding Space Math — Theory & Implementations

This notebook covers the mathematics of embedding spaces in LLMs:
1. **Embedding matrix** — lookup, parameter counting
2. **Similarity metrics** — dot product, cosine similarity, Euclidean distance, norms
3. **Geometric properties** — analogy arithmetic, isotropy analysis, curse of dimensionality
4. **Positional encodings** — sinusoidal, learned, RoPE, ALiBi
5. **Embedding training** — initialisation, gradient flow, tied weights
6. **Contextual vs static** — word2vec vs transformer representations
7. **Dimensionality reduction** — PCA, t-SNE visualisation
8. **Embedding space in attention** — Q/K/V projections, residual stream

All code uses only `numpy` and `matplotlib` (no external ML frameworks required).

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.figsize'] = (10, 6)
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — text-based outputs only")

---
## 1. Embedding Matrix — Lookup & Parameter Counting

The embedding matrix $E \in \mathbb{R}^{N \times d}$ maps token IDs to continuous vectors.

Lookup: $\text{embed}(i) = E_i$ (the $i$-th row of $E$).

Mathematically equivalent to: $\text{embed}(i) = E^\top \mathbf{e}_i$ where $\mathbf{e}_i$ is the one-hot vector.

In [ ]:
# === 1.1 Build an embedding matrix and perform lookups ===

# Vocabulary and dimensions
vocab = ["<pad>", "<unk>", "the", "cat", "sat", "on", "mat", "dog", "ran", "fast"]
N = len(vocab)      # vocabulary size
d = 8               # embedding dimension

# Initialise embedding matrix: Xavier / Glorot uniform
scale = np.sqrt(1.0 / d)
E = np.random.uniform(-scale, scale, size=(N, d))

print(f"Vocabulary size  N = {N}")
print(f"Embedding dim    d = {d}")
print(f"Embedding matrix E ∈ ℝ^({N}×{d})  — {N * d:,} parameters")
print(f"Init scale (1/√d) = {scale:.4f}\n")

# Lookup by index (row slicing)
token_id = 3  # "cat"
e_cat = E[token_id]
print(f"embed('{vocab[token_id]}') = {np.round(e_cat, 4)}")

# Lookup by one-hot multiplication (equivalent)
one_hot = np.zeros(N)
one_hot[token_id] = 1.0
e_cat_onehot = E.T @ one_hot

print(f"E^T @ one_hot    = {np.round(e_cat_onehot, 4)}")
print(f"Match: {np.allclose(e_cat, e_cat_onehot)}")

In [ ]:
# === 1.2 Parameter counting for real models ===

models = [
    ("GPT-2 Small",    50257,   768,  "124 M"),
    ("GPT-2 Medium",   50257,  1024,  "355 M"),
    ("GPT-2 Large",    50257,  1280,  "774 M"),
    ("GPT-2 XL",       50257,  1600,  "1.5 B"),
    ("GPT-3 (175B)",   50257, 12288,  "175 B"),
    ("LLaMA-7B",       32000,  4096,  "7 B"),
    ("LLaMA-65B",      32000,  8192,  "65 B"),
]

print(f"{'Model':<18} {'Vocab N':>8} {'Dim d':>6} {'Embed Params':>14} {'Total Params':>13} {'Embed %':>8}")
print("-" * 75)
for name, n, dim, total_str in models:
    embed_params = n * dim
    # Parse total params for percentage calculation
    total_str_clean = total_str.replace(" ", "")
    if "B" in total_str_clean:
        total_val = float(total_str_clean.replace("B", "")) * 1e9
    else:
        total_val = float(total_str_clean.replace("M", "")) * 1e6
    pct = embed_params / total_val * 100
    print(f"{name:<18} {n:>8,} {dim:>6} {embed_params:>14,} {total_str:>13} {pct:>7.1f}%")

---
## 2. Similarity Metrics in Embedding Space

Three key metrics for comparing embedding vectors:

| Metric | Formula | Range | Use in LLMs |
|--------|---------|-------|-------------|
| **Dot Product** | $\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i$ | $(-\infty, +\infty)$ | Attention logits, output logits |
| **Cosine Similarity** | $\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$ | $[-1, +1]$ | Semantic similarity |
| **Euclidean Distance** | $\|\mathbf{u} - \mathbf{v}\|_2 = \sqrt{\sum_i (u_i - v_i)^2}$ | $[0, +\infty)$ | Clustering, k-NN |

**Key relationship** for unit vectors ($\|\mathbf{u}\| = \|\mathbf{v}\| = 1$):

$$\|\mathbf{u} - \mathbf{v}\|^2 = 2 - 2\cos(\mathbf{u}, \mathbf{v})$$

In [ ]:
# === 2.1 Dot product, cosine similarity, Euclidean distance ===

def dot_product(u, v):
    """Inner product: u · v = Σ u_i v_i"""
    return np.sum(u * v)

def cosine_similarity(u, v):
    """cos(u,v) = (u · v) / (||u|| ||v||)"""
    num = np.sum(u * v)
    den = np.linalg.norm(u) * np.linalg.norm(v)
    return num / den if den > 0 else 0.0

def euclidean_distance(u, v):
    """||u - v||_2 = sqrt(Σ(u_i - v_i)^2)"""
    return np.linalg.norm(u - v)

# Get some embeddings from our matrix
e_cat = E[vocab.index("cat")]
e_dog = E[vocab.index("dog")]
e_mat = E[vocab.index("mat")]
e_the = E[vocab.index("the")]

pairs = [
    ("cat", "dog",  e_cat, e_dog),
    ("cat", "mat",  e_cat, e_mat),
    ("cat", "the",  e_cat, e_the),
    ("dog", "the",  e_dog, e_the),
]

print(f"{'Pair':<12} {'Dot Prod':>10} {'Cosine':>10} {'Euclidean':>10}")
print("-" * 46)
for w1, w2, u, v in pairs:
    dp = dot_product(u, v)
    cs = cosine_similarity(u, v)
    ed = euclidean_distance(u, v)
    print(f"{w1+'-'+w2:<12} {dp:>10.4f} {cs:>10.4f} {ed:>10.4f}")

In [ ]:
# === 2.2 Verify: ||u-v||^2 = 2 - 2cos(u,v) for unit vectors ===

# Normalise to unit vectors
e_cat_n = e_cat / np.linalg.norm(e_cat)
e_dog_n = e_dog / np.linalg.norm(e_dog)

sq_dist = np.sum((e_cat_n - e_dog_n) ** 2)
from_cos = 2.0 - 2.0 * cosine_similarity(e_cat_n, e_dog_n)

print("=== Unit-vector relationship ===")
print(f"||u - v||²           = {sq_dist:.6f}")
print(f"2 - 2·cos(u,v)      = {from_cos:.6f}")
print(f"Match: {np.isclose(sq_dist, from_cos)}")

# This means: for normalised embeddings, cosine similarity and
# Euclidean distance are monotonically related — ranking is identical.
print("\n→ For normalised embeddings, cosine and Euclidean rankings are EQUIVALENT.")

In [ ]:
# === 2.3 Norm comparison: L1, L2, L∞ ===

def lp_norm(x, p):
    """General L_p norm: ||x||_p = (Σ|x_i|^p)^(1/p)"""
    return np.sum(np.abs(x) ** p) ** (1.0 / p)

def linf_norm(x):
    """L∞ norm: max|x_i|"""
    return np.max(np.abs(x))

# Compare norms for all embeddings
print(f"{'Token':<8} {'||e||_1':>8} {'||e||_2':>8} {'||e||_∞':>8}")
print("-" * 36)
for i, w in enumerate(vocab):
    e = E[i]
    print(f"{w:<8} {lp_norm(e, 1):>8.4f} {lp_norm(e, 2):>8.4f} {linf_norm(e):>8.4f}")

print(f"\n--- Norm ordering inequality ---")
e_test = E[3]
l1 = lp_norm(e_test, 1)
l2 = lp_norm(e_test, 2)
linf = linf_norm(e_test)
print(f"||e||_∞ ≤ ||e||_2 ≤ ||e||_1 :  {linf:.4f} ≤ {l2:.4f} ≤ {l1:.4f}  →  {linf <= l2 <= l1}")
print(f"||e||_2 ≤ √d · ||e||_∞       :  {l2:.4f} ≤ {np.sqrt(d) * linf:.4f}  →  {l2 <= np.sqrt(d) * linf}")
print(f"||e||_1 ≤ √d · ||e||_2       :  {l1:.4f} ≤ {np.sqrt(d) * l2:.4f}  →  {l1 <= np.sqrt(d) * l2}")

---
## 3. Geometric Properties of Embedding Spaces

### 3.1 Analogy Arithmetic

The iconic word2vec result: **king − man + woman ≈ queen**

This works because semantic relationships encode as *consistent vector offsets*:

$$\vec{v}_{\text{queen}} \approx \vec{v}_{\text{king}} - \vec{v}_{\text{man}} + \vec{v}_{\text{woman}}$$

The offset $\vec{v}_{\text{king}} - \vec{v}_{\text{man}}$ captures the "gender" direction.

### 3.2 Isotropy & Anisotropy

**Isotropic**: Vectors uniformly fill the space (ideal).

**Anisotropic**: Vectors cluster in a narrow cone (common in practice).

Measured by average pairwise cosine similarity — closer to 0 is more isotropic.

In [ ]:
# === 3.1 Analogy arithmetic with synthetic embeddings ===

# Create a richer embedding space with planted semantic structure
np.random.seed(42)
d_analogy = 50
base = np.random.randn(20, d_analogy) * 0.3

# Plant semantic directions
gender_dir = np.random.randn(d_analogy)
gender_dir = gender_dir / np.linalg.norm(gender_dir)

royalty_dir = np.random.randn(d_analogy)
royalty_dir = royalty_dir / np.linalg.norm(royalty_dir)

# Create word vectors with planted structure
words = {}
words["man"]   = base[0] + 0.0 * gender_dir + 0.0 * royalty_dir
words["woman"] = base[0] + 2.0 * gender_dir + 0.0 * royalty_dir
words["king"]  = base[0] + 0.0 * gender_dir + 2.0 * royalty_dir
words["queen"] = base[0] + 2.0 * gender_dir + 2.0 * royalty_dir

words["boy"]   = base[1] + 0.0 * gender_dir + 0.0 * royalty_dir + 0.5 * np.random.randn(d_analogy)
words["girl"]  = base[1] + 2.0 * gender_dir + 0.0 * royalty_dir + 0.5 * np.random.randn(d_analogy)

words["uncle"] = base[2] + 0.0 * gender_dir
words["aunt"]  = base[2] + 2.0 * gender_dir

# Analogy: king - man + woman ≈ ?
query = words["king"] - words["man"] + words["woman"]

# Find nearest neighbor by cosine similarity
def find_nearest(query_vec, word_dict, exclude=None):
    """Find the word most similar to query_vec (by cosine), excluding certain words."""
    if exclude is None:
        exclude = set()
    best_word, best_sim = None, -np.inf
    for w, v in word_dict.items():
        if w in exclude:
            continue
        sim = cosine_similarity(query_vec, v)
        if sim > best_sim:
            best_sim = sim
            best_word = w
    return best_word, best_sim

result, sim = find_nearest(query, words, exclude={"king", "man", "woman"})
print(f"king − man + woman ≈ {result}  (cosine = {sim:.4f})")

# More analogies
analogies = [
    ("man", "woman", "king"),         # → queen
    ("man", "woman", "uncle"),        # → aunt
    ("man", "woman", "boy"),          # → girl
]

print(f"\n{'a':<8} {'b':<8} {'c':<8} {'a−b+c ≈?':<8} {'cosine':>8}")
print("-" * 44)
for a, b, c in analogies:
    q = words[a] - words[b] + words[c]  # Note: direction b → a applied to c
    # Actually the standard analogy is: a is to b as c is to ?
    # a:b :: c:? → ? = c - a + b  OR  ? = b - a + c depending on formulation
    # king:man :: queen:woman → queen = king - man + woman
    # Let's use: a - b + c → ?
    # "man" - "woman" + "king" = "queen"?  No.
    # Standard: king - man + woman = queen
    # So: a:b :: c:? means ? = c + (a - b), i.e., ? = king - man + woman
    # Let's redo properly:
    pass

# Redo clearly with standard formulation: A is to B as C is to ?
# ? ≈ C + (B − A)  ... or equivalently  ? ≈ B − A + C
print("\nAnalogy: A is to B as C is to ?")
print(f"{'A':<8} {'B':<8} {'C':<8} {'? (pred)':<8} {'cosine':>8}")
print("-" * 44)
analogy_tests = [
    ("man", "king",  "woman"),    # man:king :: woman:? → queen
    ("man", "uncle", "woman"),    # man:uncle :: woman:? → aunt
    ("man", "boy",   "woman"),    # man:boy :: woman:? → girl
]
for a, b, c in analogy_tests:
    offset = words[b] - words[a]          # relationship direction
    q = words[c] + offset                  # apply to c
    result, sim = find_nearest(q, words, exclude={a, b, c})
    print(f"{a:<8} {b:<8} {c:<8} {result:<8} {sim:>8.4f}")

In [ ]:
# === 3.2 Isotropy measurement and anisotropy simulation ===

def measure_isotropy(embeddings):
    """
    Compute average pairwise cosine similarity.
    Perfectly isotropic → avg_cos ≈ 0
    Highly anisotropic  → avg_cos ≈ 1
    """
    n = len(embeddings)
    cos_sims = []
    for i in range(n):
        for j in range(i + 1, n):
            cos_sims.append(cosine_similarity(embeddings[i], embeddings[j]))
    return np.mean(cos_sims), np.std(cos_sims)

# Case 1: Isotropic embeddings (random, well-distributed)
np.random.seed(42)
n_tokens, d_iso = 200, 128
iso_embeddings = np.random.randn(n_tokens, d_iso)
avg_cos_iso, std_cos_iso = measure_isotropy(iso_embeddings)

# Case 2: Anisotropic embeddings (common mean + small perturbation)
common_direction = np.random.randn(d_iso) * 5.0  # strong shared component
aniso_embeddings = common_direction + np.random.randn(n_tokens, d_iso) * 0.3
avg_cos_aniso, std_cos_aniso = measure_isotropy(aniso_embeddings)

print("=== Isotropy Analysis ===")
print(f"{'Type':<15} {'Avg Cosine':>12} {'Std':>8} {'Diagnosis':>15}")
print("-" * 54)
print(f"{'Isotropic':<15} {avg_cos_iso:>12.4f} {std_cos_iso:>8.4f} {'✓ Good':>15}")
print(f"{'Anisotropic':<15} {avg_cos_aniso:>12.4f} {std_cos_aniso:>8.4f} {'✗ Degenerate':>15}")

print(f"\nIsotropic avg cosine should be ≈ 0 (random directions in high-d space).")
print(f"Anisotropic avg cosine ≈ {avg_cos_aniso:.3f} (vectors clustered in a narrow cone).")

In [ ]:
# === 3.3 Fix anisotropy: mean centering and whitening ===

def mean_center(embeddings):
    """Remove the mean vector → push avg cosine toward 0."""
    mean = np.mean(embeddings, axis=0)
    return embeddings - mean

def whiten(embeddings):
    """
    ZCA whitening: decorrelate + equalise variance in all directions.
    X_white = (X - μ) @ Σ^{-1/2}
    """
    centered = embeddings - np.mean(embeddings, axis=0)
    cov = np.cov(centered, rowvar=False)  # d × d
    # Eigendecompose
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Regularise small eigenvalues
    eigvals = np.maximum(eigvals, 1e-8)
    # Σ^{-1/2} = V @ diag(1/√λ) @ V^T
    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals))
    W = eigvecs @ D_inv_sqrt @ eigvecs.T
    return centered @ W

# Apply fixes to anisotropic embeddings
centered = mean_center(aniso_embeddings)
whitened = whiten(aniso_embeddings)

avg_cos_centered, _ = measure_isotropy(centered)
avg_cos_whitened, _ = measure_isotropy(whitened)

print("=== Anisotropy Fixes ===")
print(f"{'Method':<20} {'Avg Cosine':>12} {'Improvement':>14}")
print("-" * 50)
print(f"{'Original':<20} {avg_cos_aniso:>12.4f} {'baseline':>14}")
print(f"{'Mean-centered':<20} {avg_cos_centered:>12.4f} {avg_cos_aniso - avg_cos_centered:>13.4f}↓")
print(f"{'Whitened (ZCA)':<20} {avg_cos_whitened:>12.4f} {avg_cos_aniso - avg_cos_whitened:>13.4f}↓")

In [ ]:
# === 3.4 Curse of dimensionality: distance concentration ===

def distance_concentration(dims, n_pairs=5000):
    """
    In high dimensions, distances between random points concentrate
    around the same value. The ratio (d_max - d_min) / d_mean → 0.
    
    This makes nearest-neighbor search harder.
    """
    results = []
    for d in dims:
        # Random unit vectors
        X = np.random.randn(n_pairs, d)
        X = X / np.linalg.norm(X, axis=1, keepdims=True)
        Y = np.random.randn(n_pairs, d)
        Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)
        
        dists = np.linalg.norm(X - Y, axis=1)
        ratio = (np.max(dists) - np.min(dists)) / np.mean(dists)
        cv = np.std(dists) / np.mean(dists)  # coefficient of variation
        results.append((d, np.mean(dists), np.std(dists), ratio, cv))
    return results

np.random.seed(42)
dims = [2, 10, 50, 128, 512, 1024, 4096]
results = distance_concentration(dims)

print("=== Distance Concentration in High Dimensions ===")
print(f"{'Dim d':>6} {'Mean Dist':>10} {'Std Dist':>10} {'(max-min)/mean':>16} {'CV':>8}")
print("-" * 54)
for d, mean, std, ratio, cv in results:
    print(f"{d:>6} {mean:>10.4f} {std:>10.4f} {ratio:>16.4f} {cv:>8.4f}")

print("\n→ As d ↑, the coefficient of variation (CV) shrinks.")
print("  All pairwise distances converge to √2 (for unit vectors).")
print("  This is WHY cosine similarity is preferred over Euclidean in LLMs.")

---
## 4. Positional Encodings

Without positional information, a Transformer treats input as a **set** (permutation equivariant).
Positional encodings inject order information into the embedding space.

### Methods covered:
1. **Sinusoidal PE** (Vaswani et al., 2017) — fixed, frequency-based
2. **Learned PE** (BERT, GPT-2) — trained position embedding table
3. **RoPE** (Su et al., 2021) — rotation in 2D subspaces
4. **ALiBi** (Press et al., 2022) — attention bias, no position embeddings

In [ ]:
# === 4.1 Sinusoidal Positional Encoding (Vaswani et al., 2017) ===
#
# PE(pos, 2i)   = sin(pos / 10000^(2i/d))
# PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
#
# Each dimension i has a different frequency (wavelength).
# Low i  → high frequency (short wavelengths, distinguish nearby positions)
# High i → low frequency  (long wavelengths, distinguish distant positions)

def sinusoidal_pe(max_len, d_model):
    """
    Compute sinusoidal positional encoding matrix.
    
    Returns: PE ∈ ℝ^{max_len × d_model}
    """
    PE = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]      # (max_len, 1)
    
    # Compute div_term: 10000^(2i/d) for i = 0, 1, ..., d/2 - 1
    i = np.arange(0, d_model, 2)[np.newaxis, :]       # (1, d/2)
    div_term = 10000.0 ** (i / d_model)                # (1, d/2)
    
    # angle = pos / 10000^(2i/d)
    angle = position / div_term                         # (max_len, d/2)
    
    PE[:, 0::2] = np.sin(angle)  # even indices
    PE[:, 1::2] = np.cos(angle)  # odd indices
    
    return PE

# Generate PE for a sequence
max_len, d_model = 128, 64
PE = sinusoidal_pe(max_len, d_model)

print(f"Positional Encoding matrix: {PE.shape}")
print(f"\nPE[0, :8]  (position 0): {np.round(PE[0, :8], 4)}")
print(f"PE[1, :8]  (position 1): {np.round(PE[1, :8], 4)}")
print(f"PE[50, :8] (position 50): {np.round(PE[50, :8], 4)}")

# Verify: each PE vector has unit-ish norm
norms = np.linalg.norm(PE, axis=1)
print(f"\n||PE[pos]||_2 — min: {norms.min():.4f}, max: {norms.max():.4f}, mean: {norms.mean():.4f}")
print(f"Expected: √(d/2) = {np.sqrt(d_model/2):.4f}  (since sin²+cos² = 1 per pair)")

In [ ]:
# === 4.2 Key property: PE encodes relative position via inner product ===
#
# For sinusoidal PE, the inner product PE[pos+k] · PE[pos] depends only on k,
# not on the absolute position pos. This is by design.
#
# Proof: PE[pos+k] = M(k) · PE[pos] where M(k) is a rotation/linear-map
# that depends only on the offset k.

# Verify empirically: PE[p+k]·PE[p] is approximately constant for fixed k
print("=== Relative position property of sinusoidal PE ===\n")

for k in [1, 5, 20]:
    dots = []
    for pos in range(0, max_len - k, 1):
        dot = np.dot(PE[pos], PE[pos + k])
        dots.append(dot)
    dots = np.array(dots)
    print(f"offset k={k:>3}:  mean(PE[p+k]·PE[p]) = {dots.mean():>8.4f}  "
          f"std = {dots.std():>8.6f}  "
          f"{'✓ constant' if dots.std() < 0.01 else '~ approximately constant'}")

# Show the decay of inner product with distance
print(f"\n--- Inner product vs distance (PE[0] · PE[k]) ---")
print(f"{'k':>4} {'PE[0]·PE[k]':>12}")
print("-" * 18)
for k in [0, 1, 2, 5, 10, 20, 50, 100, 127]:
    if k < max_len:
        dot = np.dot(PE[0], PE[k])
        print(f"{k:>4} {dot:>12.4f}")

In [ ]:
# === 4.3 Visualise sinusoidal PE (if matplotlib available) ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # (a) Heatmap of PE matrix
    ax = axes[0]
    im = ax.imshow(PE[:64, :32].T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xlabel('Position')
    ax.set_ylabel('Dimension')
    ax.set_title('Sinusoidal PE (first 64 pos × 32 dims)')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    # (b) Individual sinusoids for selected dimensions
    ax = axes[1]
    positions = np.arange(max_len)
    for dim_idx in [0, 1, 4, 10, 30, 62]:
        ax.plot(positions, PE[:, dim_idx], label=f'dim {dim_idx}', alpha=0.7)
    ax.set_xlabel('Position')
    ax.set_ylabel('PE value')
    ax.set_title('PE curves for selected dimensions')
    ax.legend(fontsize=7)
    
    # (c) Cosine similarity matrix between positions
    ax = axes[2]
    PE_norm = PE / np.linalg.norm(PE, axis=1, keepdims=True)
    cos_mat = PE_norm[:64] @ PE_norm[:64].T
    im = ax.imshow(cos_mat, cmap='viridis', vmin=-0.2, vmax=1.0)
    ax.set_xlabel('Position')
    ax.set_ylabel('Position')
    ax.set_title('Cosine similarity between positions')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available — skipping PE visualisation")

In [ ]:
# === 4.4 RoPE — Rotary Position Embedding (Su et al., 2021) ===
#
# RoPE applies position-dependent ROTATIONS to pairs of dimensions.
# For dimension pair (2i, 2i+1), position m gets rotated by angle m·θ_i
# where θ_i = 10000^(-2i/d).
#
# The rotation matrix for pair i at position m:
#   R_i(m) = [[cos(m·θ_i), -sin(m·θ_i)],
#             [sin(m·θ_i),  cos(m·θ_i)]]
#
# KEY PROPERTY: q_m^T · k_n depends only on (m - n), not on absolute positions.
# Proof: R_i(m)^T R_i(n) = R_i(n - m)  (rotation group property)

def rope_frequencies(d_model, base=10000.0):
    """Compute θ_i = 1 / base^(2i/d) for i = 0, ..., d/2 - 1"""
    i = np.arange(0, d_model, 2)
    theta = 1.0 / (base ** (i / d_model))
    return theta

def apply_rope(x, position, theta):
    """
    Apply RoPE rotation to vector x at given position.
    
    x:        (d,) vector
    position: scalar position index
    theta:    (d/2,) frequency array
    
    Returns: rotated x
    """
    d = len(x)
    x_rot = np.copy(x)
    
    for i in range(d // 2):
        angle = position * theta[i]
        cos_a = np.cos(angle)
        sin_a = np.sin(angle)
        
        # Apply 2×2 rotation to pair (x[2i], x[2i+1])
        x0 = x[2 * i]
        x1 = x[2 * i + 1]
        x_rot[2 * i]     = cos_a * x0 - sin_a * x1
        x_rot[2 * i + 1] = sin_a * x0 + cos_a * x1
    
    return x_rot

# Demo
d_rope = 16
theta = rope_frequencies(d_rope)

np.random.seed(42)
q = np.random.randn(d_rope)
k = np.random.randn(d_rope)

print(f"RoPE frequencies θ: {np.round(theta, 6)}")
print(f"\nFrequency spectrum spans {theta[-1]:.6f} (slow) to {theta[0]:.6f} (fast)")

# Apply RoPE at different positions
q_m = apply_rope(q, position=10, theta=theta)
k_n = apply_rope(k, position=7,  theta=theta)

print(f"\nOriginal q·k           = {np.dot(q, k):.6f}")
print(f"RoPE(q,10) · RoPE(k,7) = {np.dot(q_m, k_n):.6f}")
print(f"→ Dot product changes because RoPE encodes relative position (10-7=3)")

In [ ]:
# === 4.5 RoPE: Verify relative position property ===
#
# If RoPE works correctly: dot(RoPE(q, m), RoPE(k, n)) should be
# the same for all (m, n) pairs with the same offset m - n.

print("=== RoPE Relative Position Verification ===\n")
print(f"{'(m, n)':<12} {'offset':>7} {'dot(q_m, k_n)':>14}")
print("-" * 36)

# Test: for fixed offset Δ = 3, try different absolute positions
for m, n in [(3, 0), (10, 7), (50, 47), (100, 97)]:
    q_m = apply_rope(q, m, theta)
    k_n = apply_rope(k, n, theta)
    dot_val = np.dot(q_m, k_n)
    print(f"({m:>3}, {n:>3})   {m-n:>7}   {dot_val:>14.6f}")

print("\n✓ All entries with offset = 3 have IDENTICAL dot products!")

# Compare with different offsets
print(f"\n{'offset':>7} {'dot product':>14}")
print("-" * 24)
for delta in [0, 1, 2, 3, 5, 10, 50]:
    m, n = 20 + delta, 20
    q_m = apply_rope(q, m, theta)
    k_n = apply_rope(k, n, theta)
    print(f"{delta:>7}   {np.dot(q_m, k_n):>14.6f}")
print("\n→ Dot product varies with offset, not absolute position.")

In [ ]:
# === 4.6 ALiBi — Attention with Linear Biases (Press et al., 2022) ===
#
# ALiBi does NOT modify embeddings. Instead, it adds a linear bias
# to the attention score based on distance:
#
#   a_{ij} = q_i · k_j - m · |i - j|
#
# where m is a head-specific slope.
# Slopes are geometric: m_h = 2^{-8/H · h} for h = 1, ..., H

def alibi_slopes(n_heads):
    """Compute ALiBi slopes: m_h = 2^(-8h/H) for h = 1, ..., H"""
    return np.array([2.0 ** (-8.0 * h / n_heads) for h in range(1, n_heads + 1)])

def alibi_bias_matrix(seq_len, slope):
    """
    Compute ALiBi bias matrix for one head.
    bias[i,j] = -slope * |i - j|
    """
    positions = np.arange(seq_len)
    # Distance matrix
    dist = np.abs(positions[:, None] - positions[None, :])
    return -slope * dist

# Demo: 8 attention heads
n_heads = 8
slopes = alibi_slopes(n_heads)
print("=== ALiBi Head Slopes ===")
print(f"{'Head':>5} {'Slope m':>12} {'Effective range':>16}")
print("-" * 36)
for h, m in enumerate(slopes):
    # "effective range": how far until bias = -5 (soft cutoff)
    eff_range = int(5.0 / m) if m > 0 else float('inf')
    print(f"{h+1:>5} {m:>12.6f} {eff_range:>16}")

# Show bias matrix for one head
seq_len = 12
bias = alibi_bias_matrix(seq_len, slopes[0])
print(f"\nALiBi bias matrix (head 1, slope={slopes[0]:.4f}), seq_len={seq_len}:")
print(f"(showing bias for first 8 positions)")
for i in range(min(8, seq_len)):
    row = " ".join(f"{bias[i, j]:>6.2f}" for j in range(min(8, seq_len)))
    print(f"  pos {i}: [{row}]")

In [ ]:
# === 4.7 Positional encoding comparison ===

# Compare all four methods on key properties
print("=" * 78)
print("POSITIONAL ENCODING COMPARISON")
print("=" * 78)
print(f"{'Property':<28} {'Sinusoidal':<14} {'Learned':<14} {'RoPE':<14} {'ALiBi':<14}")
print("-" * 78)
comparisons = [
    ("Parameters",             "0",           "T × d",       "0",           "0 (H slopes)"),
    ("Relative position",      "Indirect",    "No",          "Yes (exact)", "Yes (bias)"),
    ("Length generalisation",   "Good",        "Poor",        "Good",        "Excellent"),
    ("Applied to",             "Embeddings",  "Embeddings",  "Q, K only",   "Attn scores"),
    ("Used in",                "Transformer", "BERT, GPT-2", "LLaMA, PaLM", "BLOOM"),
    ("Computational cost",     "O(T·d)",      "O(T·d)",      "O(T·d)",      "O(T²·H)"),
    ("Extrapolation to 2×T",   "Moderate",    "Fails",       "Moderate",    "Strong"),
    ("Implementation",         "Simple",      "Simplest",    "Moderate",    "Simple"),
]
for prop, *vals in comparisons:
    print(f"{prop:<28}", end="")
    for v in vals:
        print(f" {v:<14}", end="")
    print()

---
## 5. Embedding Training Dynamics

How embeddings are initialised, how gradients flow to them, and weight tying.

### Initialisation
Xavier/Glorot: $E_{ij} \sim \mathcal{U}\left(-\frac{1}{\sqrt{d}}, \frac{1}{\sqrt{d}}\right)$

This ensures $\text{Var}(E^\top \mathbf{e}_i) = \frac{1}{3d} \cdot d \cdot \frac{1}{d} = \frac{1}{3}$ — keeping activations stable.

### Weight Tying
The same matrix $E$ serves as both input embeddings and output projection:
$$\text{logit}_i = \mathbf{h}^\top \mathbf{e}_i$$
This saves $N \times d$ parameters and acts as a regulariser.

In [ ]:
# === 5.1 Initialisation: Xavier variance analysis ===
#
# Xavier uniform: U(-1/√d, 1/√d)
# Variance of one element: (2/√d)² / 12 = 4/(12d) = 1/(3d)
# Variance of one embedding vector (d elements): d × 1/(3d) = 1/3
# ⇒ ||e||² ≈ d/3  ⇒ ||e|| ≈ √(d/3)

dims_to_test = [64, 128, 256, 512, 768, 1024, 4096]
n_samples = 1000

print("=== Xavier Init Variance Analysis ===")
print(f"{'Dim d':>6} {'Theory Var':>12} {'Empirical Var':>14} {'Theory ||e||':>14} {'Empirical ||e||':>16}")
print("-" * 66)

np.random.seed(42)
for d_test in dims_to_test:
    scale = 1.0 / np.sqrt(d_test)
    samples = np.random.uniform(-scale, scale, size=(n_samples, d_test))
    
    # Element variance
    elem_var_theory = 1.0 / (3.0 * d_test)
    elem_var_empirical = np.var(samples)
    
    # Vector norm
    norm_theory = np.sqrt(d_test / 3.0)
    norm_empirical = np.mean(np.linalg.norm(samples, axis=1))
    
    print(f"{d_test:>6} {elem_var_theory:>12.6f} {elem_var_empirical:>14.6f} "
          f"{norm_theory:>14.4f} {norm_empirical:>16.4f}")

In [ ]:
# === 5.2 Gradient flow: how token frequency affects embedding learning ===
#
# In next-token prediction, the gradient of CE loss w.r.t. embedding e_i is:
#   ∂L/∂e_i = (only non-zero when token i appears in the batch)
#
# Frequent tokens get updated much more often than rare tokens.
# This creates a "frequency bias" in embedding quality.

def simulate_embedding_training(vocab_size, d_model, n_steps, token_freqs, lr=0.01):
    """
    Simplified simulation: each step, a token is sampled by frequency,
    and its embedding gets a gradient update.
    Track how many updates each token receives.
    """
    # Init embeddings
    E_sim = np.random.randn(vocab_size, d_model) * 0.01
    update_counts = np.zeros(vocab_size, dtype=int)
    
    # Normalise frequencies to probabilities
    probs = token_freqs / token_freqs.sum()
    
    for step in range(n_steps):
        # Sample a token by frequency
        token_id = np.random.choice(vocab_size, p=probs)
        update_counts[token_id] += 1
        
        # Simulated gradient (random direction, magnitude ∝ 1/freq for balance)
        grad = np.random.randn(d_model) * 0.1
        E_sim[token_id] -= lr * grad
    
    return E_sim, update_counts

# Zipf-distributed frequencies (realistic for language)
np.random.seed(42)
V = 100
token_freqs = np.array([1.0 / (i + 1) for i in range(V)])  # Zipf's law

E_trained, update_counts = simulate_embedding_training(V, 32, n_steps=50000, token_freqs=token_freqs)

print("=== Token Frequency vs Gradient Updates ===")
print(f"{'Rank':>5} {'Freq (norm)':>12} {'Updates':>8} {'% of total':>12} {'||e|| final':>12}")
print("-" * 54)
total_updates = update_counts.sum()
for rank in [0, 1, 2, 4, 9, 19, 49, 99]:
    freq_norm = token_freqs[rank] / token_freqs.sum()
    pct = update_counts[rank] / total_updates * 100
    norm = np.linalg.norm(E_trained[rank])
    print(f"{rank+1:>5} {freq_norm:>12.4f} {update_counts[rank]:>8} {pct:>11.2f}% {norm:>12.4f}")

print(f"\nTop-10 tokens receive {update_counts[:10].sum()/total_updates*100:.1f}% of all updates.")
print(f"Bottom-50 tokens receive {update_counts[50:].sum()/total_updates*100:.1f}% of all updates.")
print(f"→ Rare token embeddings are poorly trained. This motivates subword tokenisation.")

In [ ]:
# === 5.3 Weight tying: shared embedding & output projection ===
#
# Without tying:
#   Input:  E_in ∈ ℝ^{N × d}  (token → hidden)
#   Output: E_out ∈ ℝ^{d × N}  (hidden → logit)
#   Total: 2 × N × d parameters
#
# With tying:  E_out = E_in^T
#   Total: 1 × N × d parameters (50% savings on embedding parameters)
#
# Logit for token i: logit_i = h^T · e_i  (cosine-like scoring)

np.random.seed(42)
V_tie = 1000
d_tie = 256

# Separate embeddings (no tying)
E_in  = np.random.randn(V_tie, d_tie) * (1.0 / np.sqrt(d_tie))
E_out = np.random.randn(d_tie, V_tie) * (1.0 / np.sqrt(d_tie))

# Tied embeddings
E_tied = np.random.randn(V_tie, d_tie) * (1.0 / np.sqrt(d_tie))

# Simulate a hidden state
h = np.random.randn(d_tie) * 0.5

# Compute logits both ways
logits_separate = E_out.T @ h       # using separate output matrix  (N,)
logits_tied     = E_tied @ h         # using tied (input) matrix     (N,)

# Softmax
def softmax(x):
    x_shifted = x - np.max(x)
    exp_x = np.exp(x_shifted)
    return exp_x / exp_x.sum()

probs_separate = softmax(logits_separate)
probs_tied     = softmax(logits_tied)

print("=== Weight Tying Analysis ===\n")
params_separate = 2 * V_tie * d_tie
params_tied     = 1 * V_tie * d_tie
print(f"Separate:  {params_separate:>12,} embedding parameters")
print(f"Tied:      {params_tied:>12,} embedding parameters")
print(f"Savings:   {params_separate - params_tied:>12,} parameters ({(params_separate - params_tied)/params_separate*100:.0f}%)")

print(f"\nLogit stats (separate): mean={logits_separate.mean():.4f}, std={logits_separate.std():.4f}")
print(f"Logit stats (tied):     mean={logits_tied.mean():.4f}, std={logits_tied.std():.4f}")

print(f"\nTop-5 predictions (tied):  tokens {np.argsort(probs_tied)[-5:][::-1]}")
print(f"Top-5 predictions (sep):   tokens {np.argsort(probs_separate)[-5:][::-1]}")

# In tied embedding, logit = h · e_i = ||h|| ||e_i|| cos(h, e_i)
# The model predicts the token whose embedding is most aligned with h
print(f"\n→ With tying, prediction = argmax_i cos(h, e_i)")
print(f"  The model learns to make hidden states point toward the right embedding.")

---
## 6. Contextual vs Static Embeddings

**Static** (word2vec, GloVe): One vector per word, regardless of context.
- "bank" has the SAME vector in "river bank" and "bank account"

**Contextual** (BERT, GPT): Each occurrence gets a DIFFERENT vector that depends on the surrounding tokens.
- $h_i^{(l)} = f\left(h_1^{(l-1)}, h_2^{(l-1)}, \ldots, h_T^{(l-1)}\right)$

### word2vec Skip-Gram Objective

$$\max \sum_{t} \sum_{-c \le j \le c, j \ne 0} \log P(w_{t+j} \mid w_t)$$

where $P(w_O \mid w_I) = \frac{\exp(\mathbf{v}'_{w_O} \cdot \mathbf{v}_{w_I})}{\sum_{w=1}^{V} \exp(\mathbf{v}'_w \cdot \mathbf{v}_{w_I})}$

In [ ]:
# === 6.1 Skip-gram training (simplified) ===
#
# Objective: predict context words from center word.
# Instead of full softmax, we use negative sampling:
#   L = log σ(v'_pos · v_center) + Σ_{neg} log σ(-v'_neg · v_center)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def train_skipgram(corpus_ids, vocab_size, d_model, window=2,
                   n_neg=5, lr=0.05, epochs=3):
    """
    Simplified skip-gram with negative sampling.
    
    corpus_ids: list of int token IDs
    Returns: input embeddings W_in, context embeddings W_out
    """
    np.random.seed(42)
    W_in  = np.random.randn(vocab_size, d_model) * 0.1
    W_out = np.random.randn(vocab_size, d_model) * 0.1
    
    # Token frequencies for negative sampling (unigram^0.75)
    freq = np.zeros(vocab_size)
    for t in corpus_ids:
        freq[t] += 1
    freq = freq ** 0.75
    freq /= freq.sum()
    
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        n_updates = 0
        for i in range(window, len(corpus_ids) - window):
            center = corpus_ids[i]
            
            for j in range(-window, window + 1):
                if j == 0:
                    continue
                context = corpus_ids[i + j]
                
                # Positive sample
                v_c = W_in[center]
                v_o = W_out[context]
                score = np.dot(v_c, v_o)
                loss = -np.log(sigmoid(score) + 1e-10)
                
                # Gradient for positive
                grad = (sigmoid(score) - 1.0)
                W_in[center]  -= lr * grad * v_o
                W_out[context] -= lr * grad * v_c
                
                # Negative samples
                negs = np.random.choice(vocab_size, size=n_neg, p=freq)
                for neg in negs:
                    v_n = W_out[neg]
                    score_n = np.dot(v_c, v_n)
                    loss += -np.log(sigmoid(-score_n) + 1e-10)
                    
                    grad_n = sigmoid(score_n)
                    W_in[center] -= lr * grad_n * v_n
                    W_out[neg]   -= lr * grad_n * v_c
                
                epoch_loss += loss
                n_updates += 1
        
        avg_loss = epoch_loss / max(n_updates, 1)
        losses.append(avg_loss)
        print(f"  Epoch {epoch+1}/{epochs}: avg loss = {avg_loss:.4f}")
    
    return W_in, W_out, losses

# Create a tiny corpus
toy_vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "over", "big", "small"]
toy_corpus = "the cat sat on the mat the dog ran over the big mat the small cat sat on the big dog"
corpus_ids = [toy_vocab.index(w) for w in toy_corpus.split()]

print("=== Skip-Gram Training (Negative Sampling) ===")
print(f"Vocab: {len(toy_vocab)} words, Corpus: {len(corpus_ids)} tokens\n")

W_in, W_out, losses = train_skipgram(corpus_ids, len(toy_vocab), d_model=16, epochs=5)

# Show learned similarities
print(f"\n--- Learned similarities (cosine) ---")
print(f"{'word1':<8} {'word2':<8} {'cosine':>8}")
print("-" * 28)
for w1, w2 in [("cat", "dog"), ("cat", "mat"), ("big", "small"), 
               ("sat", "ran"), ("the", "on"), ("cat", "the")]:
    i1, i2 = toy_vocab.index(w1), toy_vocab.index(w2)
    sim = cosine_similarity(W_in[i1], W_in[i2])
    print(f"{w1:<8} {w2:<8} {sim:>8.4f}")

In [ ]:
# === 6.2 Contextual embeddings: how context changes representation ===
#
# In a Transformer, after each layer, the representation of token i
# depends on ALL other tokens via self-attention.
#
# Simplified simulation: approximate contextual mixing

def simulate_contextual_embedding(static_embeds, attention_weights):
    """
    Simulate one layer of contextual processing.
    
    h_i^{(1)} = Σ_j α_{ij} × W_v × h_j^{(0)}
    
    For simplicity, W_v = I (identity).
    """
    # attention_weights: (T, T) — each row sums to 1
    return attention_weights @ static_embeds

# Create sentences with polysemous word "bank"
#   Sentence 1: "the river bank was steep"
#   Sentence 2: "the bank account was empty"
# Using our toy embeddings, simulate with made-up attention patterns

np.random.seed(42)
d_ctx = 16

# Static embeddings for words
word_embeds = {
    "the":     np.random.randn(d_ctx) * 0.3,
    "river":   np.random.randn(d_ctx) * 0.3 + np.array([1]*4 + [0]*12),   # nature direction
    "bank":    np.random.randn(d_ctx) * 0.3,                                 # ambiguous
    "was":     np.random.randn(d_ctx) * 0.3,
    "steep":   np.random.randn(d_ctx) * 0.3 + np.array([1]*4 + [0]*12),   # nature direction
    "account": np.random.randn(d_ctx) * 0.3 + np.array([0]*4 + [1]*4 + [0]*8), # finance
    "empty":   np.random.randn(d_ctx) * 0.3 + np.array([0]*4 + [1]*4 + [0]*8), # finance
}

# Sentence 1: "the river bank was steep" — bank = nature context
sent1_words = ["the", "river", "bank", "was", "steep"]
sent1_embeds = np.stack([word_embeds[w] for w in sent1_words])

# Sentence 2: "the bank account was empty" — bank = finance context
sent2_words = ["the", "bank", "account", "was", "empty"]
sent2_embeds = np.stack([word_embeds[w] for w in sent2_words])

# Simulate attention: each word attends mostly to itself + neighbors
def local_attention(T, self_weight=0.5, neighbor_weight=0.2):
    A = np.eye(T) * self_weight
    for i in range(T):
        for j in range(T):
            if i != j:
                dist = abs(i - j)
                A[i, j] = neighbor_weight / (dist + 0.5)
    # Normalize rows
    A = A / A.sum(axis=1, keepdims=True)
    return A

attn1 = local_attention(len(sent1_words))
attn2 = local_attention(len(sent2_words))

# Contextual embeddings after one "layer"
ctx1 = simulate_contextual_embedding(sent1_embeds, attn1)
ctx2 = simulate_contextual_embedding(sent2_embeds, attn2)

# Compare: static "bank" is the same; contextual "bank" differs
bank_static = word_embeds["bank"]
bank_ctx_river  = ctx1[2]  # "bank" in river context
bank_ctx_finance = ctx2[1]  # "bank" in finance context

print("=== Contextual vs Static: Polysemy Resolution ===\n")
print(f"Static 'bank' is identical in both sentences:")
print(f"  cos(bank_static, bank_static) = 1.0000\n")

print(f"Contextual 'bank' differs by context:")
cos_ctx = cosine_similarity(bank_ctx_river, bank_ctx_finance)
print(f"  cos(bank_river, bank_finance)  = {cos_ctx:.4f}  ← different representations!\n")

# Contextual bank should be closer to its co-occurring words
cos_river_ctx = cosine_similarity(bank_ctx_river, ctx1[1])   # closer to "river"
cos_acct_ctx  = cosine_similarity(bank_ctx_finance, ctx2[2]) # closer to "account"
cos_river_wrong = cosine_similarity(bank_ctx_finance, ctx1[1]) # finance-bank vs river

print(f"bank_river vs river (contextual):   cos = {cos_river_ctx:.4f}  ← high (correct context)")
print(f"bank_finance vs account (contextual): cos = {cos_acct_ctx:.4f}  ← high (correct context)")
print(f"bank_finance vs river (contextual):   cos = {cos_river_wrong:.4f}  ← lower (wrong context)")

---
## 7. Dimensionality Reduction & Embedding Visualisation

High-dimensional embeddings can be projected down for analysis:

| Method | Formula / Idea | Preserves | Complexity |
|--------|---------------|-----------|------------|
| **PCA** | Max variance directions (eigenvectors of covariance) | Global structure | $O(nd^2)$ |
| **t-SNE** | Minimise KL between high-d and low-d neighbour distributions | Local clusters | $O(n^2)$ |
| **UMAP** | Topological structure preservation via fuzzy simplicial sets | Local + some global | $O(n \log n)$ |

**PCA recap**: Find $W \in \mathbb{R}^{d \times k}$ (top-$k$ eigenvectors of covariance matrix) such that $Z = (X - \mu) W$ maximises preserved variance.

In [ ]:
# === 7.1 PCA on embeddings — from scratch ===

def pca(X, n_components=2):
    """
    Principal Component Analysis from scratch using eigendecomposition.
    
    X: (n_samples, n_features) matrix
    Returns: Z (projected), W (projection matrix), explained_var_ratio
    """
    # 1. Center the data
    mu = np.mean(X, axis=0)
    X_centered = X - mu
    
    # 2. Covariance matrix: (1/n) X^T X
    n = X_centered.shape[0]
    cov = (X_centered.T @ X_centered) / (n - 1)
    
    # 3. Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # 4. Sort by descending eigenvalue
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # 5. Select top-k components
    W = eigenvectors[:, :n_components]
    
    # 6. Project
    Z = X_centered @ W
    
    # 7. Explained variance ratio
    total_var = np.sum(eigenvalues)
    explained_var_ratio = eigenvalues[:n_components] / total_var
    
    return Z, W, eigenvalues, explained_var_ratio

# Create synthetic embedding with cluster structure
np.random.seed(42)
n_per_cluster = 50
d_pca = 64

# 4 semantic clusters
clusters = {
    "animals":  np.random.randn(n_per_cluster, d_pca) * 0.3 + np.random.randn(d_pca) * 2,
    "colors":   np.random.randn(n_per_cluster, d_pca) * 0.3 + np.random.randn(d_pca) * 2,
    "verbs":    np.random.randn(n_per_cluster, d_pca) * 0.3 + np.random.randn(d_pca) * 2,
    "numbers":  np.random.randn(n_per_cluster, d_pca) * 0.3 + np.random.randn(d_pca) * 2,
}

# Stack all embeddings
labels = []
all_embeds = []
for cluster_name, embeds in clusters.items():
    all_embeds.append(embeds)
    labels.extend([cluster_name] * n_per_cluster)
X_all = np.vstack(all_embeds)

# Run PCA
Z, W_pca, eigenvalues, explained_var = pca(X_all, n_components=2)

print("=== PCA on Synthetic Embeddings ===")
print(f"Original dims: {d_pca}, Projected dims: 2")
print(f"PC1 explains {explained_var[0]*100:.1f}% of variance")
print(f"PC2 explains {explained_var[1]*100:.1f}% of variance")
print(f"Total (2 PCs): {sum(explained_var)*100:.1f}%\n")

# Show cumulative explained variance
print("Cumulative explained variance:")
cum_var = np.cumsum(eigenvalues) / np.sum(eigenvalues)
for k in [1, 2, 5, 10, 20, 32, 64]:
    if k <= len(cum_var):
        print(f"  Top {k:>2} PCs: {cum_var[k-1]*100:>6.1f}%")

# Show cluster centroids in PCA space
print(f"\nCluster centroids in 2D PCA space:")
for i, name in enumerate(clusters.keys()):
    cluster_z = Z[i * n_per_cluster:(i + 1) * n_per_cluster]
    cx, cy = np.mean(cluster_z, axis=0)
    print(f"  {name:<10}: ({cx:>7.3f}, {cy:>7.3f})")

In [ ]:
# === 7.2 Visualise PCA clusters ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # (a) 2D PCA scatter
    ax = axes[0]
    colors_map = {"animals": "red", "colors": "blue", "verbs": "green", "numbers": "orange"}
    for i, (name, color) in enumerate(colors_map.items()):
        cluster_z = Z[i * n_per_cluster:(i + 1) * n_per_cluster]
        ax.scatter(cluster_z[:, 0], cluster_z[:, 1], c=color, label=name, alpha=0.6, s=30)
    ax.set_xlabel(f"PC1 ({explained_var[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained_var[1]*100:.1f}%)")
    ax.set_title("PCA Projection of Embedding Clusters")
    ax.legend()
    ax.grid(alpha=0.3)
    
    # (b) Scree plot (explained variance)
    ax = axes[1]
    n_show = min(30, len(eigenvalues))
    ax.bar(range(1, n_show + 1), eigenvalues[:n_show] / np.sum(eigenvalues) * 100, 
           color='steelblue', alpha=0.7)
    ax.plot(range(1, n_show + 1), cum_var[:n_show] * 100, 'r-o', markersize=3, label='Cumulative')
    ax.set_xlabel("Principal Component")
    ax.set_ylabel("Explained Variance (%)")
    ax.set_title("Scree Plot")
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available — skipping PCA visualisation")

---
## 8. Embedding Space in the Attention Mechanism

Embeddings feed directly into attention via Q/K/V projections:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

### Why $\sqrt{d_k}$ scaling?

Without scaling, $Q K^\top$ has variance $\propto d_k$ (since each element is a sum of $d_k$ products).  
Large variance → softmax saturates → vanishing gradients.  
Dividing by $\sqrt{d_k}$ normalises variance back to $O(1)$.

### Residual Stream View (Elhage et al.)

Each layer ADDS to the residual stream rather than replacing it:

$$x^{(l)} = x^{(l-1)} + \text{Attn}\left(x^{(l-1)}\right) + \text{FFN}\left(x^{(l-1)}\right)$$

The final representation is a **sum of contributions** from all layers.

In [ ]:
# === 8.1 Scaled dot-product attention from scratch ===

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute attention: softmax(QK^T / √d_k) V
    
    Q: (T_q, d_k)
    K: (T_k, d_k)
    V: (T_k, d_v)
    mask: (T_q, T_k) boolean — True = mask out (set to -inf)
    
    Returns: output (T_q, d_v), attention_weights (T_q, T_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: QK^T / √d_k
    scores = Q @ K.T / np.sqrt(d_k)
    
    # Step 2: Apply mask (for causal/padding)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    # Step 3: Softmax (row-wise)
    scores_shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores_shifted)
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Step 4: Weighted sum of values
    output = attn_weights @ V
    
    return output, attn_weights

# Demo: 5-token sequence through attention
np.random.seed(42)
T, d_model_attn, d_k, d_v = 5, 32, 16, 16

# Embeddings (input)
X = np.random.randn(T, d_model_attn) * 0.5

# Projection matrices
W_Q = np.random.randn(d_model_attn, d_k) * (1.0 / np.sqrt(d_model_attn))
W_K = np.random.randn(d_model_attn, d_k) * (1.0 / np.sqrt(d_model_attn))
W_V = np.random.randn(d_model_attn, d_v) * (1.0 / np.sqrt(d_model_attn))

# Project to Q, K, V
Q = X @ W_Q   # (T, d_k)
K = X @ W_K   # (T, d_k)
V = X @ W_V   # (T, d_v)

print(f"Input X:  {X.shape}")
print(f"Q:        {Q.shape}")
print(f"K:        {K.shape}")
print(f"V:        {V.shape}")

# No masking (bidirectional attention)
output_bidir, weights_bidir = scaled_dot_product_attention(Q, K, V)

# Causal masking (autoregressive — GPT-style)
causal_mask = np.triu(np.ones((T, T), dtype=bool), k=1)
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print(f"\n=== Attention Weights (Bidirectional) ===")
for i in range(T):
    row = " ".join(f"{weights_bidir[i, j]:.3f}" for j in range(T))
    print(f"  token {i}: [{row}]")

print(f"\n=== Attention Weights (Causal) ===")
for i in range(T):
    row = " ".join(f"{weights_causal[i, j]:.3f}" for j in range(T))
    print(f"  token {i}: [{row}]")

print(f"\n→ Causal: each token can only attend to itself and previous tokens.")
print(f"  Row sums to 1: {np.allclose(weights_causal.sum(axis=1), 1.0)}")

In [ ]:
# === 8.2 √d_k scaling: why it matters ===
#
# Without scaling, each element of QK^T is a sum of d_k products:
#   (QK^T)_{ij} = Σ_l q_{il} k_{jl}
# If q, k ~ N(0,1), each product ~ N(0,1), so the sum has variance d_k.
# Large variance → softmax pushes to one-hot → gradient vanishing.

np.random.seed(42)

print("=== Effect of √d_k Scaling on Attention ===\n")
print(f"{'d_k':>5} {'Var(QK^T) unscaled':>22} {'Var(QK^T) scaled':>20} {'Max attn (unscaled)':>22} {'Max attn (scaled)':>20}")
print("-" * 93)

for d_k_test in [4, 16, 64, 256, 1024, 4096]:
    T_test = 10
    Q_test = np.random.randn(T_test, d_k_test)
    K_test = np.random.randn(T_test, d_k_test)
    V_test = np.random.randn(T_test, d_k_test)
    
    # Unscaled
    scores_unscaled = Q_test @ K_test.T
    var_unscaled = np.var(scores_unscaled)
    
    # Scaled  
    scores_scaled = scores_unscaled / np.sqrt(d_k_test)
    var_scaled = np.var(scores_scaled)
    
    # Softmax of each
    def row_softmax(S):
        S_shifted = S - np.max(S, axis=-1, keepdims=True)
        e = np.exp(S_shifted)
        return e / e.sum(axis=-1, keepdims=True)
    
    attn_unscaled = row_softmax(scores_unscaled)
    attn_scaled   = row_softmax(scores_scaled)
    
    max_unscaled = np.max(attn_unscaled)
    max_scaled   = np.max(attn_scaled)
    
    print(f"{d_k_test:>5} {var_unscaled:>22.2f} {var_scaled:>20.4f} "
          f"{max_unscaled:>22.6f} {max_scaled:>20.6f}")

print(f"\n→ Without scaling, as d_k ↑, variance explodes and attention becomes one-hot.")
print(f"  With √d_k scaling, variance stays ≈ 1 regardless of d_k.")

In [ ]:
# === 8.3 Residual stream: accumulation across layers ===
#
# Each Transformer layer ADDS to the residual stream:
#   x^{(l)} = x^{(l-1)} + Attn(x^{(l-1)}) + FFN(x^{(l-1)})
#
# The final token representation = initial embedding + sum of all layer outputs.
# This is key for mechanistic interpretability.

def simple_ffn(x, W1, W2, b1, b2):
    """Simple 2-layer FFN with ReLU: max(0, xW1 + b1) W2 + b2"""
    hidden = np.maximum(0, x @ W1 + b1)  # ReLU
    return hidden @ W2 + b2

np.random.seed(42)
T_res, d_res = 4, 16
d_ff = 32  # FFN hidden dim
n_layers = 6

# Initial embeddings
x = np.random.randn(T_res, d_res) * 0.1

# Track contributions
layer_contributions = [np.linalg.norm(x)]
residual_norms = [np.linalg.norm(x)]

x_current = x.copy()

for layer_idx in range(n_layers):
    # Simulate attention output (simplified)
    W_Q_l = np.random.randn(d_res, d_res) * (1.0 / np.sqrt(d_res))
    W_K_l = np.random.randn(d_res, d_res) * (1.0 / np.sqrt(d_res))
    W_V_l = np.random.randn(d_res, d_res) * (1.0 / np.sqrt(d_res))
    
    Q_l = x_current @ W_Q_l
    K_l = x_current @ W_K_l
    V_l = x_current @ W_V_l
    attn_out, _ = scaled_dot_product_attention(Q_l, K_l, V_l)
    
    # Residual connection 1: x = x + Attn(x)
    x_current = x_current + attn_out * 0.1  # scale down to keep stable
    
    # FFN
    W1 = np.random.randn(d_res, d_ff) * (1.0 / np.sqrt(d_res))
    W2 = np.random.randn(d_ff, d_res) * (1.0 / np.sqrt(d_ff))
    b1 = np.zeros(d_ff)
    b2 = np.zeros(d_res)
    ffn_out = simple_ffn(x_current, W1, W2, b1, b2)
    
    # Residual connection 2: x = x + FFN(x)
    x_current = x_current + ffn_out * 0.1
    
    layer_contributions.append(np.linalg.norm(attn_out * 0.1) + np.linalg.norm(ffn_out * 0.1))
    residual_norms.append(np.linalg.norm(x_current))

print("=== Residual Stream Accumulation ===\n")
print(f"{'Layer':<8} {'||residual||':>14} {'Layer contribution':>20} {'Frac of total':>15}")
print("-" * 60)
total_contrib = sum(layer_contributions)
for i in range(n_layers + 1):
    label = "embed" if i == 0 else f"layer {i}"
    frac = layer_contributions[i] / total_contrib * 100
    print(f"{label:<8} {residual_norms[i]:>14.4f} {layer_contributions[i]:>20.4f} {frac:>14.1f}%")

print(f"\n→ The residual stream grows as each layer adds its contribution.")
print(f"  cos(initial_embedding, final_repr) = {cosine_similarity(x.flatten(), x_current.flatten()):.4f}")
print(f"  The initial embedding signal persists through all layers (skip connections).")

In [ ]:
# === 8.4 Multi-head attention ===
#
# Split d_model into H heads, each with d_k = d_model / H.
# Each head learns different attention patterns.
# Outputs are concatenated and projected back.

def multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O, mask=None):
    """
    Multi-head attention.
    
    X: (T, d_model)
    W_Qs, W_Ks, W_Vs: list of H weight matrices, each (d_model, d_k)
    W_O: (H*d_v, d_model) output projection
    
    Returns: output (T, d_model), list of per-head attention weights
    """
    head_outputs = []
    head_weights = []
    
    for h in range(len(W_Qs)):
        Q_h = X @ W_Qs[h]
        K_h = X @ W_Ks[h]
        V_h = X @ W_Vs[h]
        
        out_h, attn_h = scaled_dot_product_attention(Q_h, K_h, V_h, mask=mask)
        head_outputs.append(out_h)
        head_weights.append(attn_h)
    
    # Concatenate heads and project
    concat = np.concatenate(head_outputs, axis=-1)  # (T, H*d_v)
    output = concat @ W_O                             # (T, d_model)
    
    return output, head_weights

# Demo: 4 heads
np.random.seed(42)
T_mh, d_model_mh = 6, 32
H = 4
d_k_mh = d_model_mh // H  # 8

X_mh = np.random.randn(T_mh, d_model_mh) * 0.3

# Create per-head projection matrices
scale_init = 1.0 / np.sqrt(d_model_mh)
W_Qs = [np.random.randn(d_model_mh, d_k_mh) * scale_init for _ in range(H)]
W_Ks = [np.random.randn(d_model_mh, d_k_mh) * scale_init for _ in range(H)]
W_Vs = [np.random.randn(d_model_mh, d_k_mh) * scale_init for _ in range(H)]
W_O  = np.random.randn(H * d_k_mh, d_model_mh) * scale_init

output_mh, head_weights_mh = multi_head_attention(X_mh, W_Qs, W_Ks, W_Vs, W_O)

print(f"=== Multi-Head Attention (H={H}, d_k={d_k_mh}) ===\n")
print(f"Input:  {X_mh.shape}")
print(f"Output: {output_mh.shape}")

# Show that different heads attend to different patterns
print(f"\n--- Entropy of attention distributions per head ---")
print(f"(Higher entropy = more uniform attention; lower = more focused)")
print(f"\n{'Head':>5} {'Avg Entropy':>13} {'Max Weight':>12} {'Pattern':>20}")
print("-" * 54)
for h in range(H):
    attn = head_weights_mh[h]
    # Entropy for each query position
    entropies = -np.sum(attn * np.log(attn + 1e-10), axis=-1)
    avg_entropy = np.mean(entropies)
    max_entropy = np.log(T_mh)
    max_weight = np.max(attn)
    pattern = "uniform" if avg_entropy > 0.7 * max_entropy else "focused"
    print(f"{h+1:>5} {avg_entropy:>13.4f} {max_weight:>12.4f} {pattern:>20}")

print(f"\nMax possible entropy: {np.log(T_mh):.4f} (uniform over {T_mh} positions)")
print(f"→ Different heads can specialise in different attention patterns.")

---
## Summary

This notebook covered the **complete mathematical machinery** of embedding spaces in LLMs:

| Section | Key Concepts | Key Formulas |
|---------|-------------|--------------|
| **1. Embedding Matrix** | Lookup, parameter counting | $E \in \mathbb{R}^{N \times d}$, $\text{embed}(i) = E_i$ |
| **2. Similarity Metrics** | Dot product, cosine, Euclidean, norms | $\cos(\mathbf{u},\mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$ |
| **3. Geometric Properties** | Analogies, isotropy, curse of dimensionality | $\vec{v}_{\text{queen}} \approx \vec{v}_{\text{king}} - \vec{v}_{\text{man}} + \vec{v}_{\text{woman}}$ |
| **4. Positional Encodings** | Sinusoidal, RoPE, ALiBi | $\text{PE}(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)$ |
| **5. Training Dynamics** | Xavier init, gradient flow, weight tying | $E_{ij} \sim \mathcal{U}\left(-\frac{1}{\sqrt{d}}, \frac{1}{\sqrt{d}}\right)$ |
| **6. Contextual vs Static** | Skip-gram, polysemy resolution | $\max \sum_t \sum_j \log P(w_{t+j} \mid w_t)$ |
| **7. Dimensionality Reduction** | PCA, scree plots, clustering | $Z = (X - \mu)W_k$ |
| **8. Attention Connection** | Q/K/V, scaling, residual stream, multi-head | $\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$ |

**Next**: See `exercises.ipynb` for hands-on practice, and `../03-Attention-Math/` for deeper attention treatment.